# Limpieza del Resumen al Cumplimiento de Objetivos

Limpieza de `Analisis_GAC_Full7agosto_2026_Resumen_.csv`. Es un reporte con la misma lógica que Leads Reales: título, promedios y totales arriba, la tabla mensual real al centro, y una sección de mínimos/óptimos/máximos (`Cotas`) al final — ninguna de las dos partes es un registro individual, así que se recortan.

## 1. Cargar el archivo

In [ ]:
import pandas as pd

# Ajusta esta ruta si el archivo no esta en la misma carpeta que este notebook
archivo_entrada = "Analisis_GAC_Full7agosto_2026_Resumen_.csv"

# header=None porque la fila 1 del archivo es un titulo, no el encabezado real de la tabla
raw = pd.read_csv(archivo_entrada, encoding="utf-8-sig", header=None)
print("Forma cruda:", raw.shape)

## 2. Extraer la tabla mensual (quitar filas 1-5 y 38-49)

El encabezado real de la tabla está en la fila 6 de tu hoja (índice 5), y los datos de Febrero 2024 a Julio 2026 van de la fila 7 a la 36 (índice 6 a 35) — eso deja fuera exactamente las filas 1-5 (título, "Promedio Mes" y totales) y 38 en adelante (la fila 37 ya está vacía de por sí, y 38-49 es la sección `Cotas` de mínimos/óptimos/máximos) que pediste quitar.

También noté que, de las 61 columnas del archivo, solo 36 tienen encabezado real y datos consistentes mes a mes; el resto (columnas 34 a 55 y 58 a 60) están vacías o son notas sueltas sin encabezado — se excluyen igual que las columnas "Items" de la Bitácora de Piso.

In [ ]:
# columnas con encabezado real y datos consistentes (se excluyen las columnas sin nombre / vacias)
columnas_utiles = list(range(34)) + [56, 57]

encabezado = raw.iloc[5, columnas_utiles].str.strip().tolist()
df = raw.iloc[6:36, columnas_utiles].copy()
df.columns = encabezado
df = df.reset_index(drop=True)

print("Forma tras extraer filas y columnas:", df.shape)
df.head()

## 3. Ajustes necesarios

- **`Año` solo aparece en el primer mes de cada año** (celda combinada en el Excel original), igual que en Leads Reales — se rellena hacia abajo.
- **`Mes`** trae espacios de más en varias filas (`"Febrero "` vs `"Febrero"`) — se recortan.
- **`"-"`** se usa en varias columnas como marcador de "no aplica" — se convierte a un valor nulo real, en vez de quedar como texto.
- **`Autorizadas`, `Formalizado` y `Contado`** (las variables que vamos a usar para Sturges) se convierten a numéricas — ya venían limpias, sin comas ni símbolos, así que no perdieron ningún dato en la conversión.

No convertí a número las otras ~30 columnas (porcentajes, objetivos, montos en pesos): no las pediste y cada una tiene un formato distinto (%, $, comas de miles), así que tocarlas todas se sale del alcance de esta limpieza. Tampoco rellené los nulos restantes con "Sin dato" — son columnas numéricas (metas, proyecciones, variaciones), y meterles texto rompería cualquier cálculo posterior, igual que en Leads Reales.

In [ ]:
df["Año"] = df["Año"].ffill().astype(int)
df["Mes"] = df["Mes"].str.strip()

df = df.replace("-", pd.NA)

for columna in ["Autorizadas", "Formalizado", "Contado"]:
    df[columna] = pd.to_numeric(df[columna], errors="coerce")

print(df[["Año", "Mes", "Autorizadas", "Formalizado", "Contado"]])

## 4. Revisión final y exportar

In [ ]:
print("Forma final:", df.shape)
print("Nulos en Autorizadas/Formalizado/Contado:", df[["Autorizadas","Formalizado","Contado"]].isna().sum().tolist())
print("Filas duplicadas:", df.duplicated().sum())

archivo_salida = "Resumen_Limpia.csv"
df.to_csv(archivo_salida, index=False, encoding="utf-8-sig")
print("Archivo guardado como:", archivo_salida)

df.head()